# Phase 3 — Dataset Formatting
**Goal:** convert the cleaned records in `data/processed/cleaned/` into the final fine-tuning shape —
a **chat-turn (messages) JSONL** schema — and split it deterministically into `train` / `val` / `test`.

**Output schema (`formatting.format.schema: "chat-v1"`)**
```json
{
  "id": "fingpt-fiqa-qa-0000",
  "source": "fingpt-fiqa-qa",
  "messages": [
    {"role": "user", "content": "What is considered a business expense on a business trip?"},
    {"role": "assistant", "content": "The IRS Guidance pertaining to the subject..."}
  ]
}
```

Splitting is **deterministic** (fixed seed) and **stratified by source** so each split keeps a proportional
slice of all three sources. No model or training loop is set up here.

*Inputs:* `data/processed/cleaned/*.jsonl` · *Outputs:* `data/processed/formatted/{train,val,test}.jsonl`


In [1]:
"""Setup: paths, config, imports."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import yaml

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (c for c in [_here, *_here.parents]
     if (c / "docs").is_dir() and (c / "data").is_dir() and (c / "notebooks").is_dir()),
    _here,
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import formatting as fmt  # noqa: E402

CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
FORMATTED_DIR = PROJECT_ROOT / "data" / "processed" / "formatted"
CONFIG_PATH = PROJECT_ROOT / "configs" / "data_config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    full_cfg = yaml.safe_load(fh)
cfg = fmt.merge_config(full_cfg.get("formatting"))

print("Project root:", PROJECT_ROOT)
print("Schema:", cfg["format"]["schema"], "| system_prompt:", cfg["format"].get("system_prompt"))
print("Split ratios:", cfg["split"]["train"], cfg["split"]["val"], cfg["split"]["test"],
      "| seed:", cfg["split"]["seed"], "| stratify_by_source:", cfg["split"]["stratify_by_source"])

Project root: C:\Users\AbdElhalk\OneDrive\Desktop\NLP Projects\Financial Advice Chatbot
Schema: chat-v1 | system_prompt: None
Split ratios: 0.8 0.1 0.1 | seed: 42 | stratify_by_source: True


## 1. Read the cleaned data

In [2]:
"""Read cleaned records and summarize counts per source."""
clean_files = sorted(CLEANED_DIR.glob("*.jsonl"))
cleaned_records = []
for p in clean_files:
    cleaned_records.extend(fmt.read_cleaned(p))

by_source = pd.Series([r["source"] for r in cleaned_records]).value_counts().sort_index()
print(f"Cleaned records loaded: {len(cleaned_records)} from {len(clean_files)} sources")
print(by_source.to_string())
assert len(cleaned_records) > 0

Cleaned records loaded: 602 from 3 sources
financebench           144
fingpt-fiqa-qa         160
personal-finance-v2    298


## 2. Format to chat-turn (`chat-v1`)

In [3]:
"""Convert each cleaned record to the chat-v1 schema."""
formatted = [r for r in (fmt.format_record(rec, cfg) for rec in cleaned_records) if r is not None]
dropped = len(cleaned_records) - len(formatted)
print(f"Formatted records: {len(formatted)}  (dropped: {dropped})")

print("\nExample record (full JSON):")
print(json.dumps(formatted[0], indent=2, ensure_ascii=False))

Formatted records: 602  (dropped: 0)

Example record (full JSON):
{
  "id": "financebench-0000",
  "source": "financebench",
  "messages": [
    {
      "role": "user",
      "content": "What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement."
    },
    {
      "role": "assistant",
      "content": "$1577.00"
    }
  ]
}


## 3. Split into train / val / test
Deterministic, stratified by source (`source -> {train, val, test}` proportions replicated for every source).


In [4]:
"""Split the formatted records."""
split_cfg = cfg["split"]
trains, vals, tests = fmt.split_records(
    formatted,
    ratios=(split_cfg["train"], split_cfg["val"], split_cfg["test"]),
    seed=split_cfg["seed"],
    stratify_by_source=split_cfg["stratify_by_source"],
)

parts = {"train": trains, "val": vals, "test": tests}
print(f"Overall: {len(formatted)} records")
print(f"  train={len(trains)}  val={len(vals)}  test={len(tests)}  (sum={len(trains)+len(vals)+len(tests)})")

rows = []
for name, recs in parts.items():
    for src in sorted({r["source"] for r in recs}):
        rows.append({"split": name, "source": src, "count": sum(1 for r in recs if r["source"] == src)})
rows.append({"split": "total", "source": "all", "count": len(formatted)})
print(pd.DataFrame(rows).to_string(index=False))

Overall: 602 records
  train=481  val=61  test=60  (sum=602)
split              source  count
train        financebench    115
train      fingpt-fiqa-qa    128
train personal-finance-v2    238
  val        financebench     15
  val      fingpt-fiqa-qa     16
  val personal-finance-v2     30
 test        financebench     14
 test      fingpt-fiqa-qa     16
 test personal-finance-v2     30
total                 all    602


## 4. Write `train` / `val` / `test` JSONL files and validate them

In [5]:
"""Persist the three split files and validate schema conformance."""
FORMATTED_DIR.mkdir(parents=True, exist_ok=True)
for name, recs in parts.items():
    path = FORMATTED_DIR / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for rec in recs:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Wrote files:")
for name in ("train", "val", "test"):
    path = FORMATTED_DIR / f"{name}.jsonl"
    print(f"   {path}  ({sum(1 for _ in path.open(encoding='utf-8'))} lines)")

print("\nSchema validation (chat-v1):")
for name in ("train", "val", "test"):
    path = FORMATTED_DIR / f"{name}.jsonl"
    n_ok, n_bad, sample = fmt.validate_file(path, schema=cfg["format"]["schema"])
    print(f"   {name:5s} valid={n_ok:4d} invalid={n_bad}")
    if sample:
        print("      ", "\n       ".join(sample))

Wrote files:
   C:\Users\AbdElhalk\OneDrive\Desktop\NLP Projects\Financial Advice Chatbot\data\processed\formatted\train.jsonl  (481 lines)
   C:\Users\AbdElhalk\OneDrive\Desktop\NLP Projects\Financial Advice Chatbot\data\processed\formatted\val.jsonl  (61 lines)
   C:\Users\AbdElhalk\OneDrive\Desktop\NLP Projects\Financial Advice Chatbot\data\processed\formatted\test.jsonl  (60 lines)

Schema validation (chat-v1):
   train valid= 481 invalid=0
   val   valid=  61 invalid=0
   test  valid=  60 invalid=0


## 5. Example formatted records

In [6]:
"""Print a few example records (one per source + val/test) to eyeball the format."""
def demo(rec) -> None:
    print(f"[{rec['id']}]")
    for m in rec["messages"]:
        content = m["content"].replace("\n", " ")
        content = content if len(content) <= 220 else content[:217] + "..."
        print(f"   {m['role']:<9}: {content}")

def first_of_source(recs, src):
    return next(r for r in recs if r["source"] == src)

print("TRAIN examples (one per source):")
for src in ("personal-finance-v2", "fingpt-fiqa-qa", "financebench"):
    demo(first_of_source(trains, src)); print()

print("VAL example:")
demo(vals[0]); print()
print("TEST example:")
demo(tests[0])

TRAIN examples (one per source):
[personal-finance-v2-0224]
   user     : 40% settlement offer from collections agency? I have a collections for 3344.62. At this point the interest is nearly the amount of the principal, so it has nearly doubled in 4 years. I would like to get this off my cr...
   assistant: Here's a thoughtful response tailored to your situation: --- **First, I want to acknowledge how frustrating this must feel**-seeing the debt grow over four years and now facing a tough decision to resolve it. You're a...

[fingpt-fiqa-qa-0234]
   user     : Financially Shielded Entity Separating Individuals Behind It From Risks
   assistant: You are describing a corporation. You can set up a corporation to perform business, but if you were using the money for any personal reasons the courts could Pierce the corporate veil and hold you personally liable. A...

[financebench-0042]
   user     : How much has the effective tax rate of American Express changed between FY2021 and FY2022?


## 6. Summary & next steps

In [7]:
"""Final summary."""
print("Final formatted dataset:")
for name in ("train", "val", "test"):
    n = len(parts[name])
    frac = n / len(formatted)
    print(f"  {name:6s} {n:4d} records ({frac:.1%})")
print(f"\nTotal: {len(formatted)} records in data/processed/formatted/ (schema: {cfg['format']['schema']})")

note = (
    "\nNext: Phase 4 will consume these files for model fine-tuning. This phase deliberately "
    "does NOT set up the model or training loop, and does not touch model/, training/, "
    "backend/, or frontend/. The system_prompt is left null so the persona can be chosen at "
    "fine-tuning time without regenerating the data."
)
print(note)

Final formatted dataset:
  train   481 records (79.9%)
  val      61 records (10.1%)
  test     60 records (10.0%)

Total: 602 records in data/processed/formatted/ (schema: chat-v1)

Next: Phase 4 will consume these files for model fine-tuning. This phase deliberately does NOT set up the model or training loop, and does not touch model/, training/, backend/, or frontend/. The system_prompt is left null so the persona can be chosen at fine-tuning time without regenerating the data.
